# Fine-Tune DistilBERT on RavenPack Headlines (TRNA Substitute)

**Goal:** adapt the PhraseBank-trained DistilBERT classifier to RavenPack news headlines, using RavenPack `event_sentiment_score` as the supervision signal (our TRNA substitute).

Companion module: `src/sentiment_ltr/models/ravenpack_sentiment.py`  
Plan reference: `docs/news_sentiment_finetuning_plan.md` (Iteration 4)

---

## Inputs

| Input | Location / detail |
| --- | --- |
| **RavenPack article export** | `data/raw/news/ravenpack/{ticker}_articles_2003_2014.parquet` — built by `notebooks/fetch_news_articles.ipynb` |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` (or `article_time`) |
| **Starting checkpoint** (recommended) | `data/models/phrasebank_distilbert_best/` — DistilBERT fine-tuned on Financial PhraseBank |
| **Fallback init** | `distilbert-base-uncased` if no PhraseBank checkpoint exists |
| **Label mapping** | `event_sentiment_score` → `negative` / `neutral` / `positive` (threshold ±0.05) |
| **Train / val / test split** | Time-based: train ≤ 2011, validation = 2012, test ≥ 2013 |
| **Hyperparameters** | `lr=2e-5`, `batch_size=16`, `max_length=128`, `epochs=2` (default) |

**Prerequisites:** conda env `sentiment-ltr-paper` with `requirements-finetuning.txt` installed; at least one ticker with a rich RavenPack export (currently **AAPL**: ~69k labeled headlines).

---

## Steps

1. **Environment check** — confirm `torch`, `transformers`, `datasets`, and GPU/MPS device.
2. **Discover data** — list `*_articles_*.parquet` under `data/raw/news/ravenpack/`; pick ticker(s).
3. **Load & inspect** — labeled rows (headline + score), class balance, split row counts.
4. **Build HF dataset** — map headlines to `sentence`, scores to 3-class `label`; time-based splits.
5. **Load init checkpoint** — PhraseBank weights (recommended) or `distilbert-base-uncased`.
6. **Tokenize** — `max_length=128`, fixed padding (same as PhraseBank pipeline).
7. **Train** — Hugging Face `Trainer`, 2 epochs, eval each epoch, `load_best_model_at_end` on **validation macro-F1**.
8. **Evaluate** — report validation + **test** accuracy and macro-F1.
9. **Save** — write checkpoint + `metrics.json` for the app / batch scoring.

---

## Expected outputs

| Output | Path / description |
| --- | --- |
| **Fine-tuned model** | `data/models/ravenpack_distilbert_best/` (`config.json`, tokenizer, weights) |
| **Training metrics** | `data/models/ravenpack_distilbert_best/metrics.json` — train loss, val/test `eval_f1`, `eval_accuracy`, hyperparams, tickers used |
| **Console summary** | Split sizes (e.g. AAPL: ~38k train / ~13k val / ~18k test), class balance, final test macro-F1 |
| **Downstream use** | Sentiment Lab → *Fine-tune on RavenPack headlines*; future batch-scoring of cached headlines (Iteration 4.2) |

**Success criteria:** training completes without error; test macro-F1 is reported; checkpoint loads for inference on new headlines.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

%load_ext autoreload
%autoreload 2

from sentiment_ltr.models.phrasebank_sentiment import (
    DEFAULT_MODEL_DIR,
    MAX_LENGTH,
    MODEL_NAME,
    label_maps,
    load_classifier,
    load_metrics,
    load_phrasebank,
    model_is_saved,
    predict_sentences,
)

N = 2  # samples per split

# 1. Load PhraseBank-trained checkpoint
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )
tokenizer, model, device = load_classifier(DEFAULT_MODEL_DIR)

# How was this model tokenized?
# - Tokenizer object: loaded from the checkpoint dir (tokenizer.json + vocab saved at train time).
# - Model config: architecture + label head only — it does NOT store padding/truncation rules.
# - Training contract: metrics.json + phrasebank_sentiment.tokenize_dataset().
metrics = load_metrics(DEFAULT_MODEL_DIR)
train_max_length = metrics.get("max_length", MAX_LENGTH)
base_model = metrics.get("model_name", MODEL_NAME)

pd.DataFrame(
    {
        "setting": [
            "base_model (HF hub id)",
            "tokenizer_class",
            "tokenizer_loaded_from",
            "model_type",
            "vocab_size",
            "padding_side",
            "tokenizer.model_max_length",
            "train/infer max_length",
            "training truncation",
            "training padding",
            "inference padding (predict_sentences)",
        ],
        "value": [
            base_model,
            type(tokenizer).__name__,
            tokenizer.name_or_path,
            model.config.model_type,
            model.config.vocab_size,
            tokenizer.padding_side,
            tokenizer.model_max_length,
            train_max_length,
            True,
            "max_length",
            "longest in batch (padding=True)",
        ],
    }
)

# 2. Sample inferences on PhraseBank validation & test (splits the model was trained on)
raw = load_phrasebank()
_, id2label, _ = label_maps(raw)

val = raw["validation"].shuffle(seed=42).select(range(N))
test = raw["test"].shuffle(seed=42).select(range(N))

samples = pd.concat(
    [
        pd.DataFrame({
            "split": "validation",
            "sentence": val["sentence"],
            "label": [id2label[int(i)] for i in val["label"]],
        }),
        pd.DataFrame({
            "split": "test",
            "sentence": test["sentence"],
            "label": [id2label[int(i)] for i in test["label"]],
        }),
    ],
    ignore_index=True,
)

preds = predict_sentences(samples["sentence"].tolist(), tokenizer, model, device)

# Training-style tokenization for the first sample (fixed length 128)
example = samples["sentence"].iloc[0]
enc = tokenizer(
    example,
    truncation=True,
    padding="max_length",
    max_length=train_max_length,
)
pd.DataFrame(
    {
        "token": tokenizer.convert_ids_to_tokens(enc["input_ids"]),
        "input_id": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
    }
)

samples.join(preds.drop(columns=["sentence"])).assign(
    match=lambda df: df["label"] == df["pred"]
)


In [ ]:
from sentiment_ltr.models.phrasebank_sentiment import phrasebank_probability_chart_frame
from sentiment_ltr.viz import split_series_distribution_figures

CLASS_ORDER = ["negative", "neutral", "positive"]

long_probs = phrasebank_probability_chart_frame()
split_order = long_probs["split"].cat.categories.tolist()
chart_orders = {"split": split_order, "class": CLASS_ORDER}
chart_labels = {
    "split": "PhraseBank split",
    "probability": "Predicted probability",
    "p50": "Median predicted probability",
    "class": "Class",
}

fig_box, fig_p50, p50 = split_series_distribution_figures(
    long_probs,
    x="split",
    y="probability",
    series="class",
    category_orders=chart_orders,
    axis_labels=chart_labels,
    box_title="Class probabilities by split (box & whisker)",
    median_title="Median class probability by split (p50)",
    median_col="p50",
    x_hover_label="PhraseBank split",
    series_hover_label="Class",
    y_hover_label="Predicted probability",
    median_y_hover_label="Median predicted probability",
)
fig_box.show()
fig_p50.show()


### How was original model tokenized

### Schema of the labels the original model was trained on

In [ ]:
from transformers import AutoConfig

from sentiment_ltr.models.ravenpack_sentiment import ID2LABEL as RP_ID2LABEL

# Label schema baked into the PhraseBank checkpoint (starting weights for RavenPack)
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )

pb_config = AutoConfig.from_pretrained(DEFAULT_MODEL_DIR)
checkpoint_id2label = {int(k): v for k, v in pb_config.id2label.items()}
checkpoint_label2id = {int(k): v for k, v in pb_config.label2id.items()}

raw = load_phrasebank()
_, dataset_id2label, dataset_label2id = label_maps(raw)

print("PhraseBank checkpoint config (model.config)")
display(
    pd.DataFrame(
        {"id": list(checkpoint_id2label.keys()), "label": list(checkpoint_id2label.values())}
    ).set_index("id")
)
print("label2id:", checkpoint_label2id)

print("\nPhraseBank HF dataset ClassLabel (should match checkpoint)")
display(
    pd.DataFrame(
        {"id": list(dataset_id2label.keys()), "label": list(dataset_id2label.values())}
    ).set_index("id")
)

print("\nRavenPack fine-tune target labels (same 3-class schema)")
display(
    pd.DataFrame(
        {"id": list(RP_ID2LABEL.keys()), "label": list(RP_ID2LABEL.values())}
    ).set_index("id")
)

assert checkpoint_id2label == dataset_id2label == RP_ID2LABEL
print("\n✓ All three id2label maps match — RavenPack labels align with PhraseBank head.")


## RavenPack inputs & outputs (AAPL)

Loads the rich local export from `notebooks/fetch_news_articles.ipynb`:

| Stage | What | Detail |
| --- | --- | --- |
| **Input file** | `data/raw/news/ravenpack/aapl_articles_2003_2014.parquet` | ~311k article rows × 23 columns (headline, scores, prices, metadata) |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` | Used by `load_ravenpack_labeled_frame()` |
| **Label rule** | `event_sentiment_score` → 3 classes | `> +0.05` positive, `< -0.05` negative, else neutral |
| **Labeled frame** | Deduped rows with `label_name` + `label` | ~69k usable headlines for fine-tuning |
| **HF output** | `DatasetDict` with `sentence` + `label` | Time split: train ≤2011, val 2012, test ≥2013 |

Module: `src/sentiment_ltr/models/ravenpack_sentiment.py`


In [ ]:
from IPython.display import display

from sentiment_ltr.models.ravenpack_sentiment import (
    RAVENPACK_NEWS_DIR,
    SENTIMENT_SCORE_THRESHOLD,
    SPLIT_SOURCE,
    discover_ravenpack_article_files,
    load_ravenpack_labeled_frame,
    ravenpack_class_balance,
    ravenpack_split_summary,
    ravenpack_to_hf_dataset,
)

TICKER = "AAPL"

# ── Input: local parquet export ─────────────────────────────────────────────
article_paths = discover_ravenpack_article_files([TICKER])
if not article_paths:
    raise FileNotFoundError(
        f"No RavenPack export for {TICKER} under {RAVENPACK_NEWS_DIR}. "
        "Run notebooks/fetch_news_articles.ipynb first."
    )
article_path = article_paths[0]

print("INPUT")
print(f"  file:        {article_path.relative_to(PROJECT_ROOT)}")
print(f"  label rule:  |event_sentiment_score| > {SENTIMENT_SCORE_THRESHOLD} → pos/neg; else neutral")
print(f"  splits:      {SPLIT_SOURCE}\n")

raw_rp = pd.read_parquet(article_path)
display(
    pd.Series(
        {
            "rows": f"{len(raw_rp):,}",
            "columns": raw_rp.shape[1],
            "date_min": str(raw_rp["article_date"].min()),
            "date_max": str(raw_rp["article_date"].max()),
        },
        name="raw export summary",
    ).to_frame("value")
)
print("Raw column dtypes:")
display(raw_rp.dtypes.to_frame("dtype"))
print("Sample raw rows (training-relevant columns):")
display(
    raw_rp[
        ["ticker", "article_date", "headline", "event_sentiment_score", "relevance_score", "story_id"]
    ].head(5)
)

# ── Labeled frame (module output) ─────────────────────────────────────────
labeled = load_ravenpack_labeled_frame([TICKER])

print("\nLABELED FRAME (after score→label, headline filter, dedupe)")
display(
    pd.Series(
        {
            "rows": f"{len(labeled):,}",
            "dropped_from_raw": f"{len(raw_rp) - len(labeled):,}",
            "columns": labeled.shape[1],
        },
        name="labeled summary",
    ).to_frame("value")
)
display(ravenpack_class_balance(labeled))
display(ravenpack_split_summary(labeled))
print("Sample labeled rows:")
display(
    labeled[
        ["ticker", "article_date", "headline", "event_sentiment_score", "label_name", "label"]
    ].head(5)
)

# ── Hugging Face DatasetDict (fine-tuning input) ────────────────────────────
hf_ds = ravenpack_to_hf_dataset(labeled)

print("\nHF DATASET OUTPUT (what train_ravenpack() consumes)")
hf_summary = pd.DataFrame(
    {
        "split": list(hf_ds.keys()),
        "rows": [len(hf_ds[s]) for s in hf_ds],
        "columns": [", ".join(hf_ds[s].column_names) for s in hf_ds],
    }
)
display(hf_summary)
print("Sample train rows (`sentence` = headline, `label` = 0/1/2):")
display(hf_ds["train"].to_pandas().head(5))


## AAPL inference: predicted vs RavenPack actual labels

Score headlines with the best available checkpoint (RavenPack if trained, else PhraseBank) and compare `pred` to RavenPack `label_name` (`event_sentiment_score` → negative / neutral / positive).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from sentiment_ltr.models.ravenpack_sentiment import (
    DEFAULT_RAVENPACK_MODEL_DIR,
    ID2LABEL,
    assign_time_split,
    ravenpack_model_is_saved,
)

# ── Config ──────────────────────────────────────────────────────────────────
# Out-of-box = PhraseBank checkpoint applied to RavenPack headlines (no RavenPack fine-tune)
FORCE_CHECKPOINT = "phrasebank"  # "phrasebank" | "ravenpack" | "auto"
EVAL_SPLIT = "test"  # train | validation | test | None
BATCH_SIZE = 64
MAX_ROWS = None  # e.g. 2000 for a quick smoke test

if FORCE_CHECKPOINT == "phrasebank":
    model_dir = DEFAULT_MODEL_DIR
    ckpt_label = "PhraseBank (out-of-box on RavenPack)"
elif FORCE_CHECKPOINT == "ravenpack":
    if not ravenpack_model_is_saved():
        raise FileNotFoundError("No RavenPack checkpoint — train first or use phrasebank")
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned"
else:
    use_ravenpack_ckpt = ravenpack_model_is_saved()
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR if use_ravenpack_ckpt else DEFAULT_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned" if use_ravenpack_ckpt else "PhraseBank (out-of-box)"

print(f"Checkpoint: {ckpt_label}")
print(f"  path: {model_dir.relative_to(PROJECT_ROOT)}")

tokenizer, model, device = load_classifier(model_dir)

eval_df = labeled.copy()
eval_df["split"] = assign_time_split(eval_df["article_date"])
if EVAL_SPLIT:
    eval_df = eval_df[eval_df["split"] == EVAL_SPLIT].reset_index(drop=True)
if MAX_ROWS and len(eval_df) > MAX_ROWS:
    eval_df = eval_df.sample(MAX_ROWS, random_state=42).reset_index(drop=True)

print(f"\nScoring {len(eval_df):,} {TICKER} headlines" + (f" ({EVAL_SPLIT} split)" if EVAL_SPLIT else ""))

headlines = eval_df["headline"].tolist()
pred_chunks: list[pd.DataFrame] = []
for start in range(0, len(headlines), BATCH_SIZE):
    batch = headlines[start : start + BATCH_SIZE]
    pred_chunks.append(predict_sentences(batch, tokenizer, model, device))
preds = pd.concat(pred_chunks, ignore_index=True)

results = eval_df[
    ["split", "article_date", "headline", "event_sentiment_score", "label_name"]
].join(preds.drop(columns=["sentence"]))
results = results.rename(columns={"label_name": "actual"})
results["match"] = results["actual"] == results["pred"]

label_order = [ID2LABEL[i] for i in sorted(ID2LABEL)]
y_true = results["actual"]
y_pred = results["pred"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0)

print("\n=== Performance metrics ===")
print(f"  accuracy:  {acc:.1%}")
print(f"  macro-F1:  {macro_f1:.1%}")
print("\nClassification report (actual = RavenPack score labels):")
print(classification_report(y_true, y_pred, labels=label_order, digits=3, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=label_order),
    index=pd.Index(label_order, name="actual"),
    columns=pd.Index(label_order, name="pred"),
)
cm_pct = cm.div(cm.sum(axis=1), axis=0).mul(100)

display(
    cm.style.format("{:,.0f}")
    .bar(align="mid", color=["red", "lightgreen"])
    .set_caption(f"Confusion matrix — counts ({ckpt_label}, {TICKER} {EVAL_SPLIT})")
)
display(
    cm_pct.round(1)
    .style.format("{:.1f}%")
    .bar(align="mid", color=["red", "lightgreen"], vmin=0, vmax=100)
    .set_caption("Confusion matrix — % of actual class (row-normalized)")
)

print("Random mismatches:")
mismatches = results.loc[~results["match"]]
display(
    mismatches.sample(min(10, len(mismatches)), random_state=42)[
        [
            "article_date",
            "headline",
            "event_sentiment_score",
            "actual",
            "pred",
            "p(negative)",
            "p(neutral)",
            "p(positive)",
        ]
    ]
)